# Catchups - Taxonomy project

In [83]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [84]:
import pandas as pd
pd.set_option('max_colwidth', 300)

![image.png](image.png)

### **Updated Pipeline**

---

#### **Step 1: Input data preparation**
1. **Data inputs:**
   - Project abstracts or descriptions.
   - Extracted keywords (from DBPedia, RAKE, YAKE, KeyBERT, etc.).
   - Taxonomy labels, including hierarchical structure.

In [85]:
projects = catalog.load("gtr.projects.documents")

[02/13/25 12:39:11] INFO     Loading data from gtr.projects.documents (ParquetDataset)...       ]8;id=765180;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=743240;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\

In [86]:
projects.head(2)

,project_id,title,abstract_text,tech_abstract_text,potential_impact
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,Exploring the dark universe with quantum technologies,"The search to understand the nature of the elusive dark matter in our universe, making up 85% of its mass, is amongst the highest scientific priorities around the world. Terrestrial experiments focus efforts on Weakly Interacting Massive Particles (WIMPs) with ultra-low background experiments, o...",None,None
1,00022364-C7A7-4016-BEA5-29C8D160F674,Exploring the role of vitamin transport in insect models of disease vector biology,"Vector-borne diseases of plants, livestock and humans are infections transmitted by arthropods feeding on host plants or animals. These diseases have huge worldwide economic, social and health costs.\nMicronutrients such as vitamins are essential for all forms of animal life. However, many inver...",None,None



2. **Hierarchical concatenation of taxonomy labels:**
   - Transform the taxonomy into **hierarchically concatenated labels**:
     - For each bottom-level node $ l_j $, concatenate its parent labels to create a hierarchical path:
       $$
       l_j = \text{"Parent > Child > Bottom level"}
       $$

3. **Embeddings:**
   - Compute or retrieve embeddings for:
     - **Project abstracts** (document embeddings).
     - **Keywords** (keyword embeddings).
     - **Taxonomy labels** (concatenated hierarchical labels).

In [87]:
keywords = catalog.load("keywords.gtr_data.db")

[02/13/25 12:39:55] INFO     Loading data from keywords.gtr_data.db (ParquetDataset)...         ]8;id=532529;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=481129;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\

In [88]:
keywords.head(2)

,keyword,num_annotators,project_ids,uuid
1409,absorption spectroscopy,4,"[92BE9116-6A02-4FA5-9419-A02976556B5C, F1716290-0323-471F-895E-C0EB4FCD8568, ABA68717-77E3-4AAD-8B06-A0B6A5269118, 2693BBC2-2478-4BC6-B51D-F7C01EA36550, 9B3B14D3-179B-48F2-8816-85F287F964C5, 4CAAC29F-CF99-4180-8CD2-230401CFED7D, 46CD80D7-4EBD-4BB9-AC10-5EED4CA1A90B, 940FD660-6F6D-4CA1-A36F-073AD...",d0b80fe5-7946-5bb2-b2f7-687959d43c0b
6398,abstract algebra,4,"[8D229AA2-D4AB-472A-80C7-7643B06989A3, FD260969-32BB-47FD-958C-8161A9674562, 7B125073-AEAD-4E33-BF4D-D904A82B7A86, F661EEDE-443E-4EB2-9D32-86087D9D6BC6, 2D83B313-40B9-4C7C-8661-BB9E6BB83B57, 3ED6C148-D60D-40CB-B37A-D355EB159BF8, 76178790-1834-4662-8FB7-7564B91F6FF4, B9C3F6FB-046B-4572-A811-3CBD7...",22e530e5-dcb0-5042-b21e-19d97e84ae07


In [89]:
taxonomy = catalog.load("taxonomy.cwts.full.db")

[02/13/25 12:40:57] INFO     Loading data from taxonomy.cwts.full.db (ParquetDataset)...        ]8;id=529238;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=647389;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\

In [90]:
taxonomy.head(4)

,label,id_path,level,uuid
0,Physical Sciences,3,0,b35d6e90-36e2-537d-ac76-7d64b01b4c9d
1,Physical Sciences > Earth and Planetary Sciences,3 > 19,1,c43e0cd0-c84d-5094-93b3-337c899cbfcd
2,Physical Sciences > Earth and Planetary Sciences > Geophysics,3 > 19 > 1908,2,a3e52bd2-91fe-5b52-9811-d0ea3db295a0
3,Physical Sciences > Earth and Planetary Sciences > Geophysics > Tectonic and Geochronological Evolution of Orogens,3 > 19 > 1908 > 10001,3,2d79cb09-6eeb-54a1-b75a-832ad4e601a6



---

#### **Step 2: Keyword-taxonomy similarity**
1. **Keyword similarity:**
   - For each keyword $ k_i $, compute its similarity to all taxonomy labels $ l_j $:
     $$
     S_{\text{keywords}}(k_i, l_j) = \text{cosine\_similarity}(\text{embedding}(k_i), \text{embedding}(l_j))
     $$


2. **Shortlist labels for each keyword:**
   - Identify the top-$ k $ taxonomy labels $ \{l_{j_1}, l_{j_2}, \dots, l_{j_k}\} $ for each keyword, ranked by $ S_{\text{keywords}}(k_i, l_j) $.

In [91]:
keyword_candidates = catalog.load("keywords.gtr_data.cwts_matches.intermediate")

[02/13/25 12:40:58] INFO     Loading data from keywords.gtr_data.cwts_matches.intermediate      ]8;id=320396;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=444438;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [92]:
keyword_candidates.head(2)

,document_id,taxonomy_label_id,similarity_score,shannon_entropy
0,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,bf02571a-0cb7-5b7a-8580-2360d7ae0c57,0.841424,10.963063
1,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,360d9112-4fe0-502c-912a-77677eac4cd6,0.766755,10.963063



---

#### **Step 3: Sentence-taxonomy similarity**
1. **Sentence similarity:**
   - Compute the similarity between the **project sentence embeddings** $ s_p $ for project $p$ and the shortlisted taxonomy labels $ \{l_{j_1}, \dots, l_{j_k}\} $:
     $$
     S_{\text{sentence}}(s_p, l_j) = \text{cosine\_similarity}(\text{embedding}(s_p), \text{embedding}(l_j))
     $$

2. **Normalisation for weights:**
   - For each project-label pair $(p, l_k)$, which appends all sentences $s_p$, we compute:

     - Mean similarity: $mean(p, l_k) = \frac{1}{n_p}\sum_{i=1}^{n_p} S_{\text{sentence}}(s_i, l_j)$

     - Maximum similarity: $max(p, l_k) = \max_i S_{\text{sentence}}(s_i, l_k)$

     - Match count: $count(p, l_k)$ = number of sentences matching label

   Combined score balancing frequency and strength:
   $$score(p, l_k) = \frac{1 + \log(1 + count(p, l_k))}{1 + \log(1 + n_p)} \cdot mean(p, l_k) \cdot (1 + \log(1 + max(p, l_k)))$$

  This prevents:
  - Short projects from being overly penalised
  - Long projects from dominating just due to length
  - Each project's strongest topic gets score 1.0
  - Other topics are scored relative to the strongest
  - Projects of different sizes can be compared fairly

   This approach recognises that:
  - A topic mentioned consistently across sentences is likely more relevant
  - But we shouldn't penalise focused projects that discuss fewer topics
  - Very strong individual matches should boost confidence
  - Final scores should reflect relative importance within each project

In [93]:
project_sentence_candidates = catalog.load("sentences.gtr_data.cwts_matches.intermediate")

[02/13/25 12:41:11] INFO     Loading data from sentences.gtr_data.cwts_matches.intermediate     ]8;id=626288;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=266444;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [94]:
project_sentence_candidates.head(20)

,project_id,taxonomy_label_id,score_sum,score_mean,score_max,match_count,n_sentences,similarity_score
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,007c2dd7-4827-52d2-8fac-9f9f7a162934,1.998860,0.666287,0.696138,3,11,0.682915
5,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,132eb67b-d5b3-5b44-8707-095f8cd0398f,2.889536,0.722384,0.748310,4,11,0.825698
13,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,3965b106-0435-57cd-bca6-e3c768f27a99,1.382980,0.691490,0.699644,2,11,0.624146
17,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,3d7138bc-e099-51ff-a509-9232ce8c9058,2.005107,0.668369,0.680844,3,11,0.680990
27,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,664144dd-8730-5844-be77-ba12b8ce592f,5.125317,0.732188,0.782734,7,11,1.000000
28,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,6b21a2d5-5b3b-585b-bfc2-3a1a43541e81,1.383679,0.691840,0.718791,2,11,0.629032
29,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,6cc5012e-da9e-5689-9b04-742a8b4dd3fb,3.462635,0.692527,0.729273,5,11,0.840929
39,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,89237b41-8011-59e2-a240-f3666bfa22a0,1.384619,0.692309,0.712468,2,11,0.627955
41,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,90d67522-5d95-5940-a0a0-50f7c49edfc8,1.373790,0.686895,0.721485,2,11,0.625171
48,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,ac6c9b99-bbd7-5968-a13d-799ecba7f68a,2.740338,0.685084,0.703894,4,11,0.770135


   Filtering and normalisation:
   1. Project-level quantile filtering:
      $$S_{filtered}(p) = \{s : s > Q_q(S_{score}(p))\}$$
      where $Q_q(S_{score}(p))$ is the $q$-th quantile of scores within project $p$
      
   2. Per-project score normalisation:
      $$S_{final}(p, l_k) = \frac{S_{filtered}(p, l_k)}{\max_{l_j} S_{filtered}(p, l_j)}$$
      This ensures each project's top score is 1.0 while preserving relative strengths


---

#### **Step 4: Aggregate scores to labels**

  - For each keyword $k_j$, we have raw similarity scores with labels:
    $$sim(k_j, l_k) \text{ for } l_k \in L$$

  - Valid labels are restricted to those that passed sentence filtering:
    $$L_{valid}(p) = \{l_k : l_k \in S_{final}(p)\}$$

  - Keyword scores are then defined only on this subset:

    $$K_{score}(p, l_k) = \begin{cases}
      sim(k_j, l_k) & \text{if } l_k \in L_{valid}(p) \\
      \text{undefined} & \text{otherwise}
      \end{cases}$$

  This ensures keyword matches only reinforce labels that were relevant in sentence matching.

- Filter by keyword similarity quantile threshold:
  $$K_{filtered} = \{k : K_{score}(p, l_k) \geq Q_q(K_{score})\}$$
  
  This helps only consider keyword evidence that is sufficiently strong, avoiding noise from weak keyword matches.


In [95]:
project_scores = catalog.load("projects.gtr_data.cwts_scores.detailed")

[02/13/25 12:41:25] INFO     Loading data from projects.gtr_data.cwts_scores.detailed           ]8;id=191822;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=771861;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [96]:
project_scores.head(5)

,project_id,keyword_id,taxonomy_label_id,keyword,label,similarity_score_sent,similarity_score_key,shannon_entropy
0,F1716290-0323-471F-895E-C0EB4FCD8568,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,bf02571a-0cb7-5b7a-8580-2360d7ae0c57,absorption spectroscopy,Physical Sciences > Physics and Astronomy > Radiation > X-ray Absorption Spectroscopy,0.942224,0.841424,10.963063
1,2693BBC2-2478-4BC6-B51D-F7C01EA36550,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,63d32e8c-3e26-55df-a887-9221e9983b91,absorption spectroscopy,Physical Sciences > Chemistry > Spectroscopy > Molecular Structure Determination using Rotational Spectroscopy,0.955351,0.752975,10.963063
2,2693BBC2-2478-4BC6-B51D-F7C01EA36550,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,28bc4a4d-f147-51b9-97c6-a77926592b69,absorption spectroscopy,"Life Sciences > Biochemistry, Genetics and Molecular Biology > Biophysics > Biomedical Applications of Spectroscopy Techniques",0.942590,0.747444,10.963063
3,2693BBC2-2478-4BC6-B51D-F7C01EA36550,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,61d7ee3c-f58e-5369-beb5-cb9f26e8cbd7,absorption spectroscopy,Physical Sciences > Chemistry > Spectroscopy > Diffusion Coefficients in Liquid Systems,0.912268,0.722846,10.963063
4,2693BBC2-2478-4BC6-B51D-F7C01EA36550,d0b80fe5-7946-5bb2-b2f7-687959d43c0b,1c9a2300-eac7-58ed-a486-695c790d0730,absorption spectroscopy,Physical Sciences > Chemistry > Spectroscopy > Applications of Inverse Gas Chromatography,1.000000,0.713827,10.963063



---

#### **Step 5: Final label selection using relevance drop-off**

1. **Label relevance scores:**

   - - Initial weighted combination with configurable weights $\alpha$:
  $$score(p, l_k) = \big(\alpha \cdot S_{score}(p, l_k) + (1- \alpha) \cdot K_{filtered}(k_j, l_k)\big)^2$$



   - Aggregate to project-label level:

     - Take maximum of relevance scores

     - Count unique matching keywords

     - Average entropy across matches

2. **Global Binning**:

     - Use quantiles $Q_2$ and $Q_3$ across all scores:
       ```
       if score > Q_3:      high
       if Q_2 < score ≤ Q_3: medium
       if score ≤ Q_2:      low
       ```

3. **Local (Project-Level) Binning**:
     - For each project, compute relative gaps:
       $$gap_i = \frac{score(p, l_{(i)}) - score(p, l_{(i+1)})}{score(p, l_{(i)})}$$
     - Find two largest gaps ($i^*$, $j^*$)
     - Assign bins:
       ```
       if i ≤ i*:         high
       if i* < i ≤ j*:    medium
       if i > j*:         low
       ```

4. **Final Assignment**:
     - Take conservative approach:
       $$final\_bin = min(global\_bin, local\_bin)$$


## Checking an example

In [97]:
project_example_id = "00014AFD-3C1F-410E-8D00-8FF5A7F7AF54"
project_example_id2 = "51709B42-E59C-436E-8D6B-906BFC9086E5"

In [98]:
print(projects[projects["project_id"] == project_example_id]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == project_example_id]["abstract_text"].iloc[0])

Exploring the dark universe with quantum technologies
Abstract:
The search to understand the nature of the elusive dark matter in our universe, making up 85% of its mass, is amongst the highest scientific priorities around the world. Terrestrial experiments focus efforts on Weakly Interacting Massive Particles (WIMPs) with ultra-low background experiments, operating many tonnes of target mass in deep underground sites. Such experiments have swept the bulk of the available electroweak parameter space for WIMPs and in the next decade will approach an irreducible background from coherent scattering of neutrinos - indistinguishable from WIMPs. Internationally, efforts are ramping up to explore new avenues towards the first definitive detection of galactic dark matter: quantum technologies is an exciting new frontier that may provide the breakthrough.

Levitating nanospheres, opto-mechanically held with high precision, represent targets with unprecedented sensitivity to dark matter scatteri

In [99]:
print(projects[projects["project_id"] == project_example_id2]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == project_example_id2]["abstract_text"].iloc[0])

Innovative Packaging
Abstract:
The project aim is to create an innovative packaging solution that is both environmentally friendly and boosts the commercial appeal of our new range of flavoured waters.


##### **CWTS Topics**

In [100]:
project_scores_cwts = catalog.load("projects.gtr_data.cwts_scores.aggregated")

[02/13/25 12:42:09] INFO     Loading data from projects.gtr_data.cwts_scores.aggregated         ]8;id=192057;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=761602;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [101]:
project_example_cwts = project_scores_cwts[project_scores_cwts["project_id"] == project_example_id]
project_example_cwts.head(5)

,project_id,taxonomy_label_id,label,relevance_score,similarity_score_sent,similarity_score_key,shannon_entropy,num_keywords,global_bin,local_bin,final_bin
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,007c2dd7-4827-52d2-8fac-9f9f7a162934,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Interactions of Low-Energy Electrons with Matter and Atoms",0.503677,0.682915,0.761700,10.963408,1,low,low,low
1,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,664144dd-8730-5844-be77-ba12b8ce592f,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Particle Dark Matter and Detection Methods,0.853366,1.000000,0.775817,10.964092,1,high,high,high
2,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,89237b41-8011-59e2-a240-f3666bfa22a0,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Quantum Size Effects in Metallic Nanostructures",0.454910,0.627955,0.764766,10.962531,1,low,low,low
3,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,c92e45ae-3071-504a-9723-99f56f147a19,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Neutrino Flavor Transformation and Detection,0.737645,0.870409,0.836450,10.964533,2,medium,medium,medium
4,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,d9b80782-0d1a-564e-a87d-ca8f881bb451,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Slow Light Propagation and Quantum Memory",0.479315,0.642225,0.789579,10.962531,1,low,low,low


In [102]:
project_example2_cwts = project_scores_cwts[project_scores_cwts["project_id"] == project_example_id2]
project_example2_cwts.head(5)

,project_id,taxonomy_label_id,label,relevance_score,similarity_score_sent,similarity_score_key,shannon_entropy,num_keywords,global_bin,local_bin,final_bin
297462,51709B42-E59C-436E-8D6B-906BFC9086E5,00f6984a-a7bf-59cd-9f80-a1a79cb83923,"Social Sciences > Business, Management and Accounting > Marketing > Role of Packaging Design in Consumer Behavior",0.868785,1.0,0.800254,10.964186,4,high,high,high


##### **OpenAlex Concepts**

In [103]:
# taxonomy_concepts = catalog.load("taxonomy.oa_concepts.full.db")
project_scores_concepts = catalog.load("projects.gtr_data.oa_concepts_scores.aggregated")

[02/13/25 12:42:15] INFO     Loading data from projects.gtr_data.oa_concepts_scores.aggregated  ]8;id=36868;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=41240;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [104]:
# consider the same example
project_example_concepts = project_scores_concepts[project_scores_concepts["project_id"] == project_example_id]
project_example_concepts.head(5)

,project_id,taxonomy_label_id,label,relevance_score,similarity_score_sent,similarity_score_key,shannon_entropy,num_keywords,global_bin,local_bin,final_bin
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,2fa5057b-1b25-5e15-9599-dd536a391d37,Physics > Astronomy > Dark matter > Cold dark matter,0.825054,0.924974,0.876005,10.964318,1,high,medium,medium
1,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,4d664dfa-0c43-545d-a52b-ba719bc45432,Physics > Astronomy > Dark matter > Massive particle,0.823685,0.924005,0.875669,10.964318,1,high,low,low
2,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,877aecf5-f2d5-55ed-bd44-6496e8a34116,Physics > Astrophysics > Galaxy > Cold dark matter,0.711968,0.825620,0.879038,10.964520,2,medium,low,low
3,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,9a1a220d-99db-5a3d-bf8b-c244315ee673,Physics > Quantum mechanics > Galaxy > Cold dark matter,0.756205,0.868256,0.872210,10.964722,1,medium,low,low
4,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,d9f39a59-d61c-55af-a561-455547a6c7ea,Physics > Astrophysics > Dark matter > Massive particle,0.694576,0.811697,0.875564,10.964318,1,medium,low,low


In [105]:
# consider the same packaging example
project_example2_concepts = project_scores_concepts[project_scores_concepts["project_id"] == project_example_id2]
project_example2_concepts.head(5)

,project_id,taxonomy_label_id,label,relevance_score,similarity_score_sent,similarity_score_key,shannon_entropy,num_keywords,global_bin,local_bin,final_bin
172913,51709B42-E59C-436E-8D6B-906BFC9086E5,10fc0302-8f74-500a-a910-3e3f6347c0c1,Business > Marketing > Packaging engineering,0.900221,1.0,0.849411,10.964643,1,high,high,high


##### **GOScience Concepts**

In [106]:
# taxonomy_goscience = catalog.load("taxonomy.goscience.full.db")
project_scores_goscience = catalog.load("projects.gtr_data.goscience_scores.aggregated")

[02/13/25 12:42:22] INFO     Loading data from projects.gtr_data.goscience_scores.aggregated    ]8;id=94901;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=569969;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [107]:
# consider the same example
project_example_goscience = project_scores_goscience[project_scores_goscience["project_id"] == project_example_id]
project_example_goscience.head(10)

,project_id,taxonomy_label_id,label,relevance_score,similarity_score_sent,similarity_score_key,shannon_entropy,num_keywords,global_bin,local_bin,final_bin
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,04078a36-2dce-5133-a2b9-bf202dccecc7,Electronics and photonics > Sensors > Advanced radar,0.697485,0.914102,0.681907,7.828631,2,medium,low,low
1,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,099adf75-46d3-5472-acc9-ce114ce35091,Electronics and photonics > Sensors > Quantum sensors,0.766429,0.888909,0.849351,7.825079,3,high,low,low
2,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,2a7f4f33-e2c7-5b0c-8b0d-164a5b8019f1,Electronics and photonics > Sensors > Smart dust,0.687055,0.889509,0.711211,7.829084,1,medium,low,low
3,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,2deb7d86-7c0d-5d30-8d46-c89bedb1d6a8,Quantum technologies > Quantum communications,0.847976,0.915776,0.930717,7.823077,2,high,medium,medium
4,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,31ad2342-de89-573e-b05c-78216f84282c,Electronics and photonics > Future telecoms > Quantum communications,0.733293,0.866976,0.835651,7.823077,2,medium,low,low
5,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,70817187-e297-5e81-8e46-da5da1bcc7c2,Quantum technologies > Quantum timing,0.859461,0.947600,0.887222,7.823077,2,high,high,high
6,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,9145011f-6732-54f9-8da8-2efd5d711af5,Quantum technologies > Quantum imaging,0.911090,0.979730,0.905555,7.823077,2,high,high,high
7,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,91de9ff3-9314-5afe-ada9-1788bdc0fc27,Quantum technologies > Quantum sensors,0.949477,1.000000,0.924738,7.825079,3,high,high,high
8,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,9f394dbd-9020-5ab2-929e-28732a005cca,Future computing > Future computing paradigms > Quantum computing > Quantum annealler,0.758510,0.893355,0.827384,7.823077,2,medium,low,low
9,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,aaa16ece-8f64-52ac-9109-91b9df1fd363,Quantum technologies > Quantum computing > Quantum annealler,0.780929,0.881734,0.887521,7.823077,2,high,medium,medium


In [108]:
# consider the same second example
project_example2_goscience = project_scores_goscience[project_scores_goscience["project_id"] == project_example_id2]
project_example2_goscience.head(5)

,project_id,taxonomy_label_id,label,relevance_score,similarity_score_sent,similarity_score_key,shannon_entropy,num_keywords,global_bin,local_bin,final_bin
338408,51709B42-E59C-436E-8D6B-906BFC9086E5,12e2cda4-aec7-5f38-af23-872ad9395245,Advanced materials and manufacturing > Remanufacturing,0.771202,0.978519,0.683409,7.825889,1,high,high,high


---

#### **Step 6: OpenAI API Validation**  

1. **Algorithm-Agnostic Expert Labels**  
   - The LLM evaluates projects **independently** of algorithm results.  
   - Uses **RAG** to access full taxonomy context.  
   - Assigns confidence levels (**high/medium/low**) to chosen labels.  
   - Provides a **baseline set of “true” labels** unbiased by algorithm choices.  

2. **Algorithm Result Evaluation**  
   - The LLM evaluates **labels proposed by the matching algorithm**.  
   - Performs **binary classification** (`true` or `false positive`) with explanations.  
   - Identifies **algorithmic errors and biases** by detecting misleading matches.  
   - Provides **direct feedback** on algorithm performance.  

##### **Performance Measurement**  
By combining both evaluations, we can identify:  
- **True Positives** → The algorithm proposes a **high-confidence label** that the expert agrees with.  
- **False Positives** → The algorithm proposes a **high-confidence label** that the expert rejects.  
- **False Negatives** → The expert identifies a **high-confidence label** that the algorithm missed.  

This helps estimate **precision, recall, and F1 scores** for parameter tuning.  

##### **Parameter Space**  
Key algorithmic parameters being tuned:  
- **Weighting balance** → Relative importance of **sentence-level vs. keyword-level** matching.  
- **Similarity thresholds** → Minimum required **similarity scores**.  
- **Confidence thresholds** →  
  - **Global:** Dataset-wide similarity cutoffs.  
  - **Local:** Project-specific thresholds, accounting for relative label strengths.  

In [109]:
validation_example_id = "DA007068-2FB3-477E-95BE-D0C276B1AF87"

In [110]:
print(projects[projects["project_id"] == validation_example_id]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == validation_example_id]["abstract_text"].iloc[0])

Between protection and exclusion: Separated child migrants' care relationships and caring practices
Abstract:
The promise of this project lies in generating knowledge that both analyses and provides ways to address one of the greatest global challenges of our time: the care and well-being of children affected by transnational displacement and migration. It will offer insights into the care of separated migrant children in England, starting from the premise that care is not necessarily limited to that provided by an adult or the state. Our pilot studies demonstrate that a crucial way separated migrant children survive the challenges of migration and settlement is through the care they provide and receive from other migrant children. Using creative research methods designed to involve separated migrant children and adult stakeholders in reflecting on their understandings and experiences of care, this project will not only point to 'cracks in the system' (Rosen et al., 2017) but offer ins

In [111]:
agnostic_labels = catalog.load("tuning.cwts.expert_labels.processed")

[02/13/25 12:42:29] INFO     Loading data from tuning.cwts.expert_labels.processed              ]8;id=438204;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=380940;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [112]:
agnostic_labels_example = agnostic_labels[agnostic_labels["project_id"] == validation_example_id]
agnostic_labels_example.head(5)

,project_id,label,likelihood,taxonomy_label_id
1401,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System,medium,65c70378-3a52-52c0-9cb2-8b3a66b9e70c
1403,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Law > Child Protection and Legal Frameworks,high,1ce35fbc-f0c1-5952-a559-0e610d396200
1404,DA007068-2FB3-477E-95BE-D0C276B1AF87,"Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications",high,636e283f-092a-5888-981b-55a1868ccaf5
1405,DA007068-2FB3-477E-95BE-D0C276B1AF87,"Health Sciences > Medicine > Public Health, Environmental and Occupational Health > Ethical Considerations in Medical Research Participation",medium,cd5c4a8c-26d0-5843-8509-33b36268d5a7
1406,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Sociology and Political Science > Secondary Analysis of Qualitative Data,medium,fba37c34-03f5-58b3-9192-dcee9e94e36d


In [113]:
algorithm_labels = catalog.load("tuning.cwts.scores.processed")

[02/13/25 12:42:30] INFO     Loading data from tuning.cwts.scores.processed (ParquetDataset)... ]8;id=282544;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=52473;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\

In [114]:
algorithm_labels_example = algorithm_labels[algorithm_labels["project_id"] == validation_example_id]
algorithm_labels_example.head(10)

,project_id,taxonomy_label_id,positive,explanation,label
1362,DA007068-2FB3-477E-95BE-D0C276B1AF87,65c70378-3a52-52c0-9cb2-8b3a66b9e70c,True,This label accurately describes the research project as it focuses on child welfare within the context of separated child migrants' care relationships and caring practices.,Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System
1363,DA007068-2FB3-477E-95BE-D0C276B1AF87,7cf90a13-5e42-5997-8421-b23825a5a95b,True,This label is a true positive as the project delves into transnational displacement and migration studies related to separated child migrants.,Social Sciences > Social Sciences > Demography > Transnational Diasporas and Migration Studies
1364,DA007068-2FB3-477E-95BE-D0C276B1AF87,4450b837-1eb9-573f-8874-aaba60f3a425,False,"This label is a false positive as the project does not focus on Saharan migrations specifically, but rather on the care relationships of separated child migrants in the UK.",Social Sciences > Social Sciences > Anthropology > Saharan Migrations and Transnational Connections
1365,DA007068-2FB3-477E-95BE-D0C276B1AF87,92cfe6c3-a01a-51d7-8bbb-49311611b03f,True,"This label is accurate as the project involves exploring the mental health aspects of refugees and immigrants, particularly separated child migrants.",Social Sciences > Psychology > Clinical Psychology > Mental Health of Refugees and Immigrants
1366,DA007068-2FB3-477E-95BE-D0C276B1AF87,636e283f-092a-5888-981b-55a1868ccaf5,True,"This label is a true positive as the project addresses migration, education, and policy implications in the context of separated child migrants.","Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications"
1367,DA007068-2FB3-477E-95BE-D0C276B1AF87,dd67b651-1cf1-57c0-b6d6-66206b56f5a0,False,"This label is a false positive as the project does not specifically focus on education challenges for refugee students, but rather on the care relationships of separated child migrants.",Social Sciences > Social Sciences > Education > Education Challenges for Refugee Students
1368,DA007068-2FB3-477E-95BE-D0C276B1AF87,94ab87b1-2648-5156-b932-338e861c86b1,False,"This label is a false positive as the project does not primarily examine youth employment in a global context, but rather the care dynamics of separated child migrants.",Social Sciences > Social Sciences > Demography > Youth Employment in Global Context
1369,DA007068-2FB3-477E-95BE-D0C276B1AF87,34a5f90a-e262-59b5-8f14-a3312928915f,True,"This label is accurate as the project aims to understand the impact of international migration on public health, especially concerning separated child migrants.",Social Sciences > Social Sciences > Health > Impact of International Migration on Public Health
1370,DA007068-2FB3-477E-95BE-D0C276B1AF87,b51fbe3c-bf96-5195-a545-b95008d5a520,False,"This label is a false positive as the project does not engage in participatory action research in education and social sciences, but rather focuses on the care relationships of separated child migrants.",Social Sciences > Social Sciences > Education > Participatory Action Research in Education and Social Sciences


In [116]:
validation_example_cwts = project_scores_cwts[project_scores_cwts["project_id"] == validation_example_id]
validation_example_cwts[["label", "relevance_score", "global_bin", "local_bin", "final_bin"]].head(20)

,label,relevance_score,global_bin,local_bin,final_bin
790544,Social Sciences > Social Sciences > Health > Impact of International Migration on Public Health,0.677375,medium,medium,medium
790545,Social Sciences > Social Sciences > Anthropology > Saharan Migrations and Transnational Connections,0.558828,low,low,low
790546,"Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications",0.610285,low,low,low
790547,Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System,0.855275,high,high,high
790548,Social Sciences > Social Sciences > Demography > Border Formation and Migration Dynamics,0.533071,low,low,low
790549,Social Sciences > Social Sciences > Demography > Transnational Diasporas and Migration Studies,0.688077,medium,medium,medium
790550,"Social Sciences > Social Sciences > Sociology and Political Science > Migration, Borders, and Social Justice",0.676511,medium,medium,medium
790551,Social Sciences > Psychology > Clinical Psychology > Mental Health of Refugees and Immigrants,0.857344,high,high,high
790552,Social Sciences > Social Sciences > Demography > Youth Employment in Global Context,0.418558,low,low,low
790553,Social Sciences > Social Sciences > Political Science and International Relations > European Union Immigration and Asylum Policies,0.679979,medium,medium,medium
